# 0. The final model and its settings

The model is defined by one file, `config/final_model.json`; the code sets no scientific value.
This notebook shows that configuration, which values still await confirmation from the workstation
final settings run, and the openCARP commands it will produce. Nothing is simulated here.

**Order of the notebooks**

| Notebook | Stages | What it does |
| --- | --- | --- |
| 01 | preflight, conditioning, simulation | openCARP runs (hours on the workstation) |
| 02 | physiology, window | activation checks; measurement window |
| 03 | electrograms | unipolar phi_e at every electrode; forward-model check |
| 04 | measurements, changePoints | calibration, recordings, PP, clinical change points |
| 05 | groundTruth | exact contributing-tissue fields |
| 06 | report | acceptance, report.md, figures |

The same stages run in one command: `python -m efov run --out runs/v3_reference --np 24`.
Every stage is resumable and refuses outputs made with different settings.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

## The configuration, section by section

In [ ]:
for section in ["tissue", "lesion", "bath", "pacing", "numerics", "electrodes"]:
    values = {k: v for k, v in config[section].items() if not k.startswith("_")}
    display(Markdown(f"**{section}**"), pd.Series(values, dtype=object).to_frame("value"))

## Provenance: where each consequential value comes from

`awaitingWorkstation` values are the expected rule outcomes; replace them with the workstation's `productionModel.json` (next cell).

In [ ]:
pd.DataFrame(config["provenance"]).T.drop(index="_about", errors="ignore")

In [ ]:
# Import the rule-chosen values from the final settings run (pilot_v2: 3D_Code/runs/final_v2/productionModel.json):
# from efov.config import importProductionModel
# importProductionModel("path/to/productionModel.json", CONFIG, outputPath=CONFIG)

## Size of the run

In [ ]:
from efov import config as cfg
print(f"mesh: {cfg.meshNodeCount(config):,} nodes (tissue + bath)")
print(f"placements: {len(cfg.centreYmm(config)) * len(config['electrodes']['gapsMm'])}")
print(f"prepacing restart: {cfg.restartTimeMs(config)} ms of {config['pacing']['beatCount'] * config['pacing']['cycleLengthMs']} ms")

## openCARP commands (carputils `--dry`): check them before the long run

In [ ]:
from efov import simulation
state = run.runStage("conditioning")["state"]
dry = simulation.runCondition(config, "lesion", state, run.out / "dry", NP, dry=True)["dryRun"]
print("\n".join(line for line in dry.splitlines() if line.strip())[-6000:])